# 비트코인 시계열 데이터 분석

BTC-USD 데이터를 이용하여 가격 추세,
이동평균, 수익률, 변동성을 분석하는 프로젝트입니다.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import yfinance as yf

yf.set_tz_cache_location(str(Path(".venv") / ".yfinance-cache"))

TICKER = "BTC-USD"
START_DATE = "2023-01-01"
END_DATE_INCLUSIVE = "2025-12-31"
# yfinance의 end는 배타적이므로 2025-12-31 데이터를 포함하기 위해 다음 날을 사용합니다.
DOWNLOAD_END_EXCLUSIVE = "2026-01-01"
CSV_PATH = Path("data") / "bitcoin_2023_2025.csv"

print(f"Python 실행 경로: {sys.executable}")
print(f"수집 종목: {TICKER}")
print(f"요청 기간: {START_DATE} ~ {END_DATE_INCLUSIVE}")

Python 실행 경로: C:\Users\user\ia-codyssey\.venv\Scripts\python.exe
수집 종목: BTC-USD
요청 기간: 2023-01-01 ~ 2025-12-31


In [2]:
raw_data = yf.download(
    TICKER,
    start=START_DATE,
    end=DOWNLOAD_END_EXCLUSIVE,
    auto_adjust=False,
    progress=False,
)

if raw_data.empty:
    raise RuntimeError("BTC-USD 데이터를 수집하지 못했습니다.")

print(f"yfinance 원본 컬럼 구조: {type(raw_data.columns).__name__}")
print(f"yfinance 원본 컬럼: {raw_data.columns.tolist()}")

downloaded_data = raw_data.copy()
if isinstance(downloaded_data.columns, pd.MultiIndex):
    downloaded_data.columns = downloaded_data.columns.get_level_values(0)
    print("단일 종목 분석을 위해 MultiIndex의 가격 컬럼 레벨을 사용했습니다.")

downloaded_data.index = pd.to_datetime(downloaded_data.index)
downloaded_data.index.name = "Date"
CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
if CSV_PATH.exists():
    data = pd.read_csv(CSV_PATH, parse_dates=["Date"], index_col="Date")
    print(f"기존 원본 CSV를 덮어쓰지 않고 사용합니다: {CSV_PATH.resolve()}")
else:
    data = downloaded_data
    data.to_csv(CSV_PATH, encoding="utf-8")
    print(f"CSV 저장 완료: {CSV_PATH.resolve()}")

yfinance 원본 컬럼 구조: MultiIndex
yfinance 원본 컬럼: [('Adj Close', 'BTC-USD'), ('Close', 'BTC-USD'), ('High', 'BTC-USD'), ('Low', 'BTC-USD'), ('Open', 'BTC-USD'), ('Volume', 'BTC-USD')]
단일 종목 분석을 위해 MultiIndex의 가격 컬럼 레벨을 사용했습니다.
기존 원본 CSV를 덮어쓰지 않고 사용합니다: C:\Users\user\ia-codyssey\미션07_AI_데이터분석_비트코인\data\bitcoin_2023_2025.csv


In [3]:
missing_by_column = data.isna().sum()
duplicate_rows = int(data.duplicated().sum())
dates_are_sorted = bool(data.index.is_monotonic_increasing)
date_format_is_valid = isinstance(data.index, pd.DatetimeIndex)

print(f"데이터 시작 날짜: {data.index.min().date()}")
print(f"데이터 종료 날짜: {data.index.max().date()}")
print(f"전체 데이터 개수: {len(data)}")
print(f"컬럼 목록: {data.columns.tolist()}")
print("\n데이터 상위 5개:")
print(data.head(5).to_string())
print("\n데이터 하위 5개:")
print(data.tail(5).to_string())
print("\n결측치 개수(컬럼별):")
print(missing_by_column.to_string())
print(f"결측치 총개수: {int(missing_by_column.sum())}")
print(f"중복 행 개수: {duplicate_rows}")
print("\n기본 통계 정보:")
print(data.describe().to_string())
print(f"\n날짜 시간순 정렬 여부: {dates_are_sorted}")
print(f"날짜 형식 적합 여부(DatetimeIndex): {date_format_is_valid}")
print(f"날짜 인덱스 dtype: {data.index.dtype}")

데이터 시작 날짜: 2023-01-01
데이터 종료 날짜: 2025-12-31
전체 데이터 개수: 1096
컬럼 목록: ['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume']

데이터 상위 5개:
               Adj Close         Close          High           Low          Open       Volume
Date                                                                                         
2023-01-01  16625.080078  16625.080078  16630.439453  16521.234375  16547.914062   9244361700
2023-01-02  16688.470703  16688.470703  16759.343750  16572.228516  16625.509766  12097775227
2023-01-03  16679.857422  16679.857422  16760.447266  16622.371094  16688.847656  13903079207
2023-01-04  16863.238281  16863.238281  16964.585938  16667.763672  16680.205078  18421743322
2023-01-05  16836.736328  16836.736328  16884.021484  16790.283203  16863.472656  13692758566

데이터 하위 5개:
               Adj Close         Close          High           Low          Open       Volume
Date                                                                                         
2025-12

## 시계열 분석 지표

- MA20은 현재 날짜를 포함한 최근 20개 Close 가격의 산술평균입니다. 첫 19개 행은 계산에 필요한 관측치가 부족하므로 NaN입니다.
- MA60은 현재 날짜를 포함한 최근 60개 Close 가격의 산술평균입니다. 첫 59개 행은 같은 이유로 NaN입니다.
- Daily_Return은 전일 대비 Close 가격 변화율에 100을 곱한 값입니다. 첫 번째 행은 비교할 이전 날짜가 없으므로 NaN입니다.
- 30일 변동성 = 최근 30일 일별 수익률의 표준편차입니다. 연율화하지 않으며, 초반 값의 NaN은 계산 기간 부족으로 발생합니다.

이 계산 과정에서 생기는 NaN은 원본 데이터의 결측치가 아닙니다.

In [4]:
import numpy as np

analysis_data = pd.read_csv(CSV_PATH, parse_dates=["Date"], index_col="Date")
csv_dates_are_sorted = bool(analysis_data.index.is_monotonic_increasing)
csv_date_format_is_valid = isinstance(analysis_data.index, pd.DatetimeIndex)

if not csv_dates_are_sorted:
    raise ValueError("CSV의 날짜가 시간 오름차순으로 정렬되어 있지 않습니다.")

print(f"CSV 날짜 시간순 정렬 여부: {csv_dates_are_sorted}")
print(f"CSV 날짜 형식 적합 여부(DatetimeIndex): {csv_date_format_is_valid}")
print(f"원본 데이터 결측치: {int(analysis_data.isna().sum().sum())}개")
print(f"원본 데이터 중복 행: {int(analysis_data.duplicated().sum())}개")

CSV 날짜 시간순 정렬 여부: True
CSV 날짜 형식 적합 여부(DatetimeIndex): True
원본 데이터 결측치: 0개
원본 데이터 중복 행: 0개


In [5]:
analysis_data["MA20"] = analysis_data["Close"].rolling(window=20, min_periods=20).mean()
analysis_data["MA60"] = analysis_data["Close"].rolling(window=60, min_periods=60).mean()
analysis_data["Daily_Return"] = analysis_data["Close"].pct_change(fill_method=None) * 100
analysis_data["Volatility_30"] = analysis_data["Daily_Return"].rolling(window=30, min_periods=30).std()

highest_return_date = analysis_data["Daily_Return"].idxmax()
highest_return_value = analysis_data.loc[highest_return_date, "Daily_Return"]
lowest_return_date = analysis_data["Daily_Return"].idxmin()
lowest_return_value = analysis_data.loc[lowest_return_date, "Daily_Return"]
daily_return_mean = analysis_data["Daily_Return"].mean()
daily_return_std = analysis_data["Daily_Return"].std()
highest_volatility_date = analysis_data["Volatility_30"].idxmax()
highest_volatility_value = analysis_data.loc[highest_volatility_date, "Volatility_30"]

print(f"MA20 NaN 개수: {int(analysis_data['MA20'].isna().sum())}")
print(f"MA60 NaN 개수: {int(analysis_data['MA60'].isna().sum())}")
print(f"Daily_Return NaN 개수: {int(analysis_data['Daily_Return'].isna().sum())}")
print(f"Volatility_30 NaN 개수: {int(analysis_data['Volatility_30'].isna().sum())}")
print(f"가장 높은 일별 수익률: {highest_return_date.date()} / {highest_return_value:.6f}%")
print(f"가장 낮은 일별 수익률: {lowest_return_date.date()} / {lowest_return_value:.6f}%")
print(f"일별 수익률 평균: {daily_return_mean:.6f}%")
print(f"일별 수익률 표준편차: {daily_return_std:.6f}%")
print(f"30일 변동성 최고 날짜와 값: {highest_volatility_date.date()} / {highest_volatility_value:.6f}%")

MA20 NaN 개수: 19
MA60 NaN 개수: 59
Daily_Return NaN 개수: 1
Volatility_30 NaN 개수: 30
가장 높은 일별 수익률: 2024-08-08 / 12.144256%
가장 낮은 일별 수익률: 2025-03-03 / -8.682040%
일별 수익률 평균: 0.181420%
일별 수익률 표준편차: 2.446365%
30일 변동성 최고 날짜와 값: 2024-03-26 / 4.429025%


In [6]:
validation_position = 100
validation_date = analysis_data.index[validation_position]

ma20_manual = analysis_data["Close"].iloc[validation_position - 19 : validation_position + 1].mean()
ma60_manual = analysis_data["Close"].iloc[validation_position - 59 : validation_position + 1].mean()
daily_return_manual = (
    analysis_data["Close"].iloc[validation_position]
    / analysis_data["Close"].iloc[validation_position - 1]
    - 1
) * 100
volatility_30_manual = analysis_data["Daily_Return"].iloc[
    validation_position - 29 : validation_position + 1
].std()

ma20_matches = np.isclose(analysis_data["MA20"].iloc[validation_position], ma20_manual, rtol=1e-12, atol=1e-12)
ma60_matches = np.isclose(analysis_data["MA60"].iloc[validation_position], ma60_manual, rtol=1e-12, atol=1e-12)
daily_return_matches = np.isclose(analysis_data["Daily_Return"].iloc[validation_position], daily_return_manual, rtol=1e-12, atol=1e-12)
volatility_30_matches = np.isclose(analysis_data["Volatility_30"].iloc[validation_position], volatility_30_manual, rtol=1e-12, atol=1e-12)

print(f"검증 샘플 날짜: {validation_date.date()}")
print(f"MA20 검증: rolling={analysis_data['MA20'].iloc[validation_position]:.12f}, 직접 계산={ma20_manual:.12f}, 일치={ma20_matches}")
print(f"MA60 검증: rolling={analysis_data['MA60'].iloc[validation_position]:.12f}, 직접 계산={ma60_manual:.12f}, 일치={ma60_matches}")
print(f"Daily_Return 검증: pct_change={analysis_data['Daily_Return'].iloc[validation_position]:.12f}, 직접 계산={daily_return_manual:.12f}, 일치={daily_return_matches}")
print(f"Volatility_30 검증: rolling std={analysis_data['Volatility_30'].iloc[validation_position]:.12f}, 직접 계산={volatility_30_manual:.12f}, 일치={volatility_30_matches}")

assert ma20_matches and ma60_matches and daily_return_matches and volatility_30_matches

검증 샘플 날짜: 2023-04-11
MA20 검증: rolling=28173.516308593749, 직접 계산=28173.516308593749, 일치=True
MA60 검증: rolling=25244.500846354167, 직접 계산=25244.500846354167, 일치=True
Daily_Return 검증: pct_change=1.962966675857, 직접 계산=1.962966675857, 일치=True
Volatility_30 검증: rolling std=3.065045552212, 직접 계산=3.065045552212, 일치=True


## 시각화

전체 기간의 데이터를 생략하지 않고 가격, 이동평균, 일별 수익률, 30일 변동성을 각각 시각화합니다.

In [7]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt

IMAGE_DIR = Path("images")
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

def format_date_axis(ax):
    locator = mdates.AutoDateLocator(minticks=6, maxticks=10)
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
    ax.grid(alpha=0.25)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(analysis_data.index, analysis_data["Close"], color="#F7931A", linewidth=1.3, label="Close")
ax.set_title("Bitcoin Price Trend (2023-2025)")
ax.set_xlabel("Date")
ax.set_ylabel("Close Price (USD)")
ax.legend()
format_date_axis(ax)
fig.tight_layout()
fig.savefig(IMAGE_DIR / "01_price_trend.png", dpi=150, bbox_inches="tight")
plt.close(fig)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(analysis_data.index, analysis_data["Close"], color="#777777", linewidth=0.9, alpha=0.7, label="Close")
ax.plot(analysis_data.index, analysis_data["MA20"], color="#1F77B4", linewidth=1.4, label="MA20")
ax.plot(analysis_data.index, analysis_data["MA60"], color="#D62728", linewidth=1.6, label="MA60")
ax.set_title("Bitcoin Close Price with 20-Day and 60-Day Moving Averages")
ax.set_xlabel("Date")
ax.set_ylabel("Price (USD)")
ax.legend()
format_date_axis(ax)
fig.tight_layout()
fig.savefig(IMAGE_DIR / "02_moving_average.png", dpi=150, bbox_inches="tight")
plt.close(fig)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(analysis_data.index, analysis_data["Daily_Return"], color="#2CA02C", linewidth=0.8, label="Daily Return")
ax.axhline(0, color="#222222", linewidth=1.0, linestyle="--", label="0% baseline")
ax.set_title("Bitcoin Daily Return (2023-2025)")
ax.set_xlabel("Date")
ax.set_ylabel("Daily Return (%)")
ax.legend()
format_date_axis(ax)
fig.tight_layout()
fig.savefig(IMAGE_DIR / "03_daily_return.png", dpi=150, bbox_inches="tight")
plt.close(fig)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(analysis_data.index, analysis_data["Volatility_30"], color="#9467BD", linewidth=1.2, label="30-Day Volatility")
ax.set_title("Bitcoin 30-Day Volatility (Non-Annualized)")
ax.set_xlabel("Date")
ax.set_ylabel("Standard Deviation of Daily Return (%)")
ax.legend()
format_date_axis(ax)
fig.tight_layout()
fig.savefig(IMAGE_DIR / "04_volatility_30.png", dpi=150, bbox_inches="tight")
plt.close(fig)

for image_path in sorted(IMAGE_DIR.glob("*.png")):
    print(f"생성 이미지: {image_path} ({image_path.stat().st_size:,} bytes)")

생성 이미지: images\01_price_trend.png (90,901 bytes)
생성 이미지: images\02_moving_average.png (142,318 bytes)
생성 이미지: images\03_daily_return.png (207,132 bytes)
생성 이미지: images\04_volatility_30.png (117,459 bytes)


## 분석 질문별 핵심 수치

외부 사건이나 원인을 사용하지 않고 현재 CSV에서 직접 계산되는 값만 확인합니다.

In [8]:
from IPython.display import display

start_date = analysis_data.index[0]
end_date = analysis_data.index[-1]
start_close = analysis_data["Close"].iloc[0]
end_close = analysis_data["Close"].iloc[-1]
total_change_amount = end_close - start_close
total_change_pct = (end_close / start_close - 1) * 100
max_close_date = analysis_data["Close"].idxmax()
max_close = analysis_data.loc[max_close_date, "Close"]
min_close_date = analysis_data["Close"].idxmin()
min_close = analysis_data.loc[min_close_date, "Close"]

price_trend_summary = pd.DataFrame(
    [
        [start_date.date(), start_close, end_date.date(), end_close, total_change_amount, total_change_pct,
         max_close_date.date(), max_close, min_close_date.date(), min_close]
    ],
    columns=["Start_Date", "Start_Close", "End_Date", "End_Close", "Change_Amount",
             "Change_Percent", "Max_Close_Date", "Max_Close", "Min_Close_Date", "Min_Close"],
)
print("전체 가격 추세 핵심 수치:")
display(price_trend_summary.round(6))

top_5_returns = analysis_data["Daily_Return"].nlargest(5).rename("Daily_Return (%)").reset_index()
bottom_5_returns = analysis_data["Daily_Return"].nsmallest(5).rename("Daily_Return (%)").reset_index()
print("일별 수익률 상위 5개:")
display(top_5_returns.round({"Daily_Return (%)": 6}))
print("일별 수익률 하위 5개:")
display(bottom_5_returns.round({"Daily_Return (%)": 6}))

highest_return_recheck = (top_5_returns.loc[0, "Date"] == pd.Timestamp("2024-08-08")) and np.isclose(
    top_5_returns.loc[0, "Daily_Return (%)"], 12.144256099404904
)
lowest_return_recheck = (bottom_5_returns.loc[0, "Date"] == pd.Timestamp("2025-03-03")) and np.isclose(
    bottom_5_returns.loc[0, "Daily_Return (%)"], -8.682040111941614
)
print(f"기존 최고 일별 수익률 결과 일치: {highest_return_recheck}")
print(f"기존 최저 일별 수익률 결과 일치: {lowest_return_recheck}")
assert highest_return_recheck and lowest_return_recheck

전체 가격 추세 핵심 수치:


,Start_Date,Start_Close,End_Date,End_Close,Change_Amount,Change_Percent,Max_Close_Date,Max_Close,Min_Close_Date,Min_Close
0,2023-01-01,16625.080078,2025-12-31,87508.828125,70883.748047,426.366356,2025-10-06,124752.53125,2023-01-01,16625.080078


일별 수익률 상위 5개:


,Date,Daily_Return (%)
0,2024-08-08,12.144256
1,2023-10-23,10.309891
2,2024-11-11,10.223523
3,2024-03-20,9.692505
4,2025-03-02,9.550453


일별 수익률 하위 5개:


,Date,Daily_Return (%)
0,2025-03-03,-8.682040
1,2024-03-19,-8.343357
2,2024-01-12,-7.581465
3,2024-08-05,-7.098648
4,2023-08-17,-7.097917


기존 최고 일별 수익률 결과 일치: True
기존 최저 일별 수익률 결과 일치: True


In [9]:
previous_ma20 = analysis_data["MA20"].shift(1)
previous_ma60 = analysis_data["MA60"].shift(1)
upward_cross = (previous_ma20 <= previous_ma60) & (analysis_data["MA20"] > analysis_data["MA60"])
downward_cross = (previous_ma20 >= previous_ma60) & (analysis_data["MA20"] < analysis_data["MA60"])

crossover_table = analysis_data.loc[upward_cross | downward_cross, ["Close", "MA20", "MA60"]].copy()
crossover_table["Cross_Type"] = np.where(upward_cross.loc[crossover_table.index], "상승 교차", "하락 교차")
crossover_table.index.name = "Date"
crossover_table = crossover_table.reset_index()

upward_cross_dates = analysis_data.index[upward_cross]
downward_cross_dates = analysis_data.index[downward_cross]
print(f"상승 교차 횟수: {int(upward_cross.sum())}")
print("상승 교차 날짜:", [date.strftime("%Y-%m-%d") for date in upward_cross_dates])
print(f"하락 교차 횟수: {int(downward_cross.sum())}")
print("하락 교차 날짜:", [date.strftime("%Y-%m-%d") for date in downward_cross_dates])
print("이동평균 교차 전체 표:")
display(crossover_table.round({"Close": 2, "MA20": 2, "MA60": 2}))
print("교차는 단기·중기 이동평균의 관계가 바뀐 시점이며 매수·매도 신호로 해석하지 않습니다.")

상승 교차 횟수: 10
상승 교차 날짜: ['2023-03-18', '2023-06-27', '2023-10-06', '2024-02-11', '2024-05-27', '2024-07-29', '2024-09-26', '2025-01-20', '2025-04-25', '2025-09-29']
하락 교차 횟수: 11
하락 교차 날짜: ['2023-03-13', '2023-05-14', '2023-08-10', '2024-01-28', '2024-04-24', '2024-06-27', '2024-08-12', '2025-01-09', '2025-02-17', '2025-08-30', '2025-10-25']
이동평균 교차 전체 표:


,Date,Close,MA20,MA60,Cross_Type
0,2023-03-13,24197.53,22646.33,22705.76,하락 교차
1,2023-03-18,26965.88,23171.01,23113.54,상승 교차
2,2023-05-14,26930.64,28303.85,28317.65,하락 교차
3,2023-06-27,30688.16,27713.92,27548.05,상승 교차
4,2023-08-10,29429.59,29355.47,29423.17,하락 교차
5,2023-10-06,27946.60,26974.55,26906.37,상승 교차
6,2024-01-28,42035.59,42323.28,42639.18,하락 교차
7,2024-02-11,48293.92,43253.27,43189.57,상승 교차
8,2024-04-24,64276.90,66379.26,66384.06,하락 교차
9,2024-05-27,69394.55,65891.92,65647.81,상승 교차


교차는 단기·중기 이동평균의 관계가 바뀐 시점이며 매수·매도 신호로 해석하지 않습니다.


In [10]:
volatility_30_mean = analysis_data["Volatility_30"].mean()
volatility_30_median = analysis_data["Volatility_30"].median()
volatility_30_max_date = analysis_data["Volatility_30"].idxmax()
volatility_30_max = analysis_data.loc[volatility_30_max_date, "Volatility_30"]
top_5_volatility = analysis_data["Volatility_30"].nlargest(5).rename("Volatility_30 (%)").reset_index()
top_5_volatility_dates = top_5_volatility["Date"].sort_values()
top_5_volatility_span_days = int((top_5_volatility_dates.iloc[-1] - top_5_volatility_dates.iloc[0]).days)
top_5_consecutive_pairs = int(top_5_volatility_dates.diff().dt.days.eq(1).sum())

volatility_max_recheck = (volatility_30_max_date == pd.Timestamp("2024-03-26")) and np.isclose(
    volatility_30_max, 4.429025243970223
)
print(f"전체 기간 평균 30일 변동성: {volatility_30_mean:.6f}%")
print(f"전체 기간 중앙값 30일 변동성: {volatility_30_median:.6f}%")
print(f"최고 30일 변동성: {volatility_30_max_date.date()} / {volatility_30_max:.6f}%")
print(f"기존 최고 변동성 결과 일치: {volatility_max_recheck}")
print("30일 변동성 상위 5개:")
display(top_5_volatility.round({"Volatility_30 (%)": 6}))
print(f"상위 5개 날짜 범위: {top_5_volatility_span_days}일, 서로 연속인 날짜 쌍: {top_5_consecutive_pairs}개")
print("상위 5개가 2024-03-22~2024-03-27에 집중되어 동일한 고변동 구간을 중복해 보여줍니다.")
assert volatility_max_recheck

전체 기간 평균 30일 변동성: 2.367130%
전체 기간 중앙값 30일 변동성: 2.315939%
최고 30일 변동성: 2024-03-26 / 4.429025%
기존 최고 변동성 결과 일치: True
30일 변동성 상위 5개:


,Date,Volatility_30 (%)
0,2024-03-26,4.429025
1,2024-03-25,4.427042
2,2024-03-24,4.393709
3,2024-03-27,4.365763
4,2024-03-22,4.359530


상위 5개 날짜 범위: 5일, 서로 연속인 날짜 쌍: 3개
상위 5개가 2024-03-22~2024-03-27에 집중되어 동일한 고변동 구간을 중복해 보여줍니다.


### 질문 1

관찰(Fact):

- Close는 2023-01-01의 16,625.08달러에서 2025-12-31의 87,508.83달러로 70,883.75달러(426.366356%) 증가했습니다.
- 기간 중 최고 Close는 2025-10-06의 124,752.53달러이고, 최저 Close는 2023-01-01의 16,625.08달러입니다. 종료 가격은 시작 가격보다 높지만 기간 중 최고 가격보다는 낮습니다.

### 질문 2

관찰(Fact):

- 가장 높은 일별 수익률은 2024-08-08의 12.144256%이고, 가장 낮은 일별 수익률은 2025-03-03의 -8.682040%입니다.
- 상위·하위 5개 날짜와 수익률은 위 표에 정리했으며, 해당 변화의 외부 원인은 다루지 않았습니다.

### 질문 3

관찰(Fact):

- MA20이 MA60 아래 또는 같은 위치에서 위로 바뀐 상승 교차는 10회, 위 또는 같은 위치에서 아래로 바뀐 하락 교차는 11회입니다.
- 상승 교차 날짜는 2023-03-18, 2023-06-27, 2023-10-06, 2024-02-11, 2024-05-27, 2024-07-29, 2024-09-26, 2025-01-20, 2025-04-25, 2025-09-29입니다.
- 하락 교차 날짜는 2023-03-13, 2023-05-14, 2023-08-10, 2024-01-28, 2024-04-24, 2024-06-27, 2024-08-12, 2025-01-09, 2025-02-17, 2025-08-30, 2025-10-25입니다. 이는 매수·매도 신호가 아니라 두 이동평균의 관계가 바뀐 시점입니다.

### 질문 4

관찰(Fact):

- 30일 변동성의 평균은 2.367130%, 중앙값은 2.315939%, 최고값은 2024-03-26의 4.429025%입니다.
- 상위 5개 날짜는 모두 2024-03-22~2024-03-27의 6일 범위에 있고, 그중 2024-03-24~2024-03-27은 연속된 날짜입니다. 따라서 상위 5개가 같은 고변동 구간을 중복해 보여줍니다.